<a href="https://colab.research.google.com/github/CMDDclass/MS697-material/blob/main/Hands-on-session5/Hands-on-session5-BO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-driven Materials Development

# Optimization Problem: The Journey Toward the Global Optimum

<img src="https://github.com/csaybar/DLcoursera/blob/master/Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/week6/images/cost.jpg?raw=1" style="width:650px;height:300px;">


## One of the Strategies for Optimization: Bayesian Optimization
Peter I. Frazier arXiv:1807.02811v1

Bayesian optimization is an approach to optimizing objective functions that take a long time (minutes or hours) to evaluate. It is best-suited for optimization over continuous domains of less than 20 dimensions, and tolerates stochastic noise in function evaluations. It builds a surrogate for the objective and quanti es the uncertainty in that surrogate using a Bayesian machine learning technique, Gaussian process regression, and then uses an acquisition function de ned from this surrogate to decide where to sample.

Bayesian optimization (BayesOpt) is a class of machine-learning-based optimization methods focused on solving the problem
$$max_{x \subseteq A} f(x)$$
where the feasible set and objective function typically have the following properties

- The input x is in $R^d$ for a value of d that is not too large. Typically d 20 in most successful applications of BayesOpt.

- The feasible set A is a simple set, in which it is easy to assess membership. Typically A is a hyper-rectangle {$x \subseteq R^d$ : $a_i \le x_i \le b_i$} or the d-dimensional simplex {$x \subseteq R_d$ : $Σ_{x_i} = 1$}.
- The objective function f is continuous. This will typically be required to model f using Gaussian process regression.

- f is expensive to evaluate in the sense that the number of evaluations that may be performed is limited, typically to a few hundred. This limitation typically arises because each evaluation takes a substantial amount of time (typically hours), but may also occur because each evaluation bears a monetary cost (e.g., from purchasing cloud computing power, or buying laboratory materials), or an opportunity cost (e.g., if evaluating f requires asking a human subject questions who will tolerate only a limited number).

- f lacks known special structure like concavity or linearity that would make it easy to optimize using techniques that leverage such structure to improve e ciency. We summarize this by saying f is a black box.

- When we evaluate f, we observe only f(x) and no rst- or second-order derivatives. This prevents the application of rst- and second-order methods like gradient descent, Newtons method, or quasiNewton methods. We refer to problems with this property as derivative-free .

- Our focus is on fnding a global rather than local optimum.

<div style="border:2px solid #444; border-left:6px solid #444;
            border-radius:8px; padding:12px 14px; background-color:#f9f9f9;">
<b>Algorithm 1</b> &nbsp; <i>Basic pseudo-code for Bayesian optimization</i>
<hr>

1. Place a Gaussian process prior on $f$.  
2. Observe $f$ at $n_0$ points according to an initial space-filling experimental design.  
   Set $n = n_0$.  
3. <b>while</b> $n \le N$ <b>do</b>  
   &nbsp;&nbsp;&nbsp;&nbsp;a. Update the posterior probability distribution on $f$ using all available data.  
   &nbsp;&nbsp;&nbsp;&nbsp;b. Let $x_n$ be a maximizer of the acquisition function over $x$,  
   where the acquisition function is computed using the current posterior distribution.  
   &nbsp;&nbsp;&nbsp;&nbsp;c. Observe $y_n = f(x_n)$.  
   &nbsp;&nbsp;&nbsp;&nbsp;d. Increment $n$.  
4. <b>end while</b>  
5. Return a solution: either the point evaluated with the largest $f(x)$,  
   or the point with the largest posterior mean.
</div>

# Ax libary
https://ax.dev/docs/why-ax

Ax is an open-source platform for adaptive experimentation, a technique used to efficiently tune parameters in complex systems.

Complex optimization problems where we wish to tune multiple parameters to improve metric performance, but the inter-parameter interactions are not fully understood, are common across various fields including machine learning, robotics, materials science, and chemistry. This category of problem is known as "black-box" optimization. The complexity of black-box optimization problems further increases if evaluations are expensive to conduct, time-consuming, or noisy.


Core concepts

- Experiment: A process of iteratively suggesting and evaluating parameters to improve some objective.

- Parameter: A variable that can be adjusted -- a collection of these form the space we are searching over during the optimization.

- Objective: The value being optimized.

- Trial: A set of parameters and the associated objective.

- Client: An object that manages the experiment and provides methods for interacting with it.

In [ ]:
!pip install ax-platform

# BO turorial using Ax : finding optima of Hartmann6 function

## Hartmann 6 function :
$$
f(\mathbf{x}) = - \sum_{i=1}^{4} \alpha_i \exp\left( - \sum_{j=1}^{6} A_{ij} (x_j - P_{ij})^2 \right)
$$

Because Ax is a black box optimizer, we can use it to optimize any arbitrary function. In this example we will minimize the Hartmann6 function, a complicated 6-dimensional function with multiple local minima. Hartmann6 is a challenging benchmark for optimization algorithms commonly used in the global optimization literature -- it tests the algorithm's ability to identify the true global minimum, rather than mistakenly converging on a local minimum. Looking at its analytic form we can see that it would be incredibly challenging to efficiently find the global minimum either by manual trial-and-error or traditional design of experiments like grid-search or random-search.


In [ ]:
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [ ]:
# Hartmann6 function
def hartmann6(x1, x2, x3, x4, x5, x6):
    alpha = np.array([1.0, 1.2, 3.0, 3.2])
    A = np.array([
        [10, 3, 17, 3.5, 1.7, 8],
        [0.05, 10, 17, 0.1, 8, 14],
        [3, 3.5, 1.7, 10, 17, 8],
        [17, 8, 0.05, 10, 0.1, 14]
    ])
    P = 10**-4 * np.array([
        [1312, 1696, 5569, 124, 8283, 5886],
        [2329, 4135, 8307, 3736, 1004, 9991],
        [2348, 1451, 3522, 2883, 3047, 6650],
        [4047, 8828, 8732, 5743, 1091, 381]
    ])

    outer = 0.0
    for i in range(4):
        inner = 0.0
        for j, x in enumerate([x1, x2, x3, x4, x5, x6]):
            inner += A[i, j] * (x - P[i, j])**2
        outer += alpha[i] * np.exp(-inner)
    return -outer

hartmann6(0.1, 0.45, 0.8, 0.25, 0.552, 1.0)

In [ ]:
# 1. Initialize the Client.
client = Client()

The Client instance can be configured with a series of Configs that define how the experiment will be run.

The Hartmann6 problem is usually evaluated on the hypercube
$x_i∈(0,1)$, so we will define six identical RangeParameterConfigs with these bounds.

You may specify additional features like parameter constraints to further refine the search space and parameter scaling to help navigate parameters with nonuniform effects.

In [ ]:
# 2. Configure where Ax will search.
# Define six float parameters x1, x2, x3, ... for the Hartmann6 function, which is typically evaluated on the unit hypercube
parameters = [
    RangeParameterConfig(
        name="x1", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="x2", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="x3", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="x4", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="x5", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="x6", parameter_type="float", bounds=(0, 1)
    ),
]

client.configure_experiment(parameters=parameters)

Now, we must configure the objective for this optimization, which we do using Client.configure_optimization. This method expects a string objective, an expression containing either a single metric to maximize, a linear combination of metrics to maximize, or a tuple of multiple metrics to jointly maximize. These expressions are parsed using SymPy. For example:

- "score" would direct Ax to maximize a metric named score

- "-loss" would direct Ax to Ax to minimize a metric named loss

- "task_0 + 0.5 * task_1" would direct Ax to maximize the sum of two task scores, downweighting task_1 by a factor of 0.5

- "score, -flops" would direct Ax to simultaneously maximize score while minimizing flops

In [ ]:
# 3. Configure a metric for Ax to target
metric_name = "hartmann6" # this name is used during the optimization loop in Step 5
objective = f"-{metric_name}" # minimization is specified by the negative sign

client.configure_optimization(objective=objective)

We will iteratively call client.get_next_trials to "ask" Ax for a parameterization to evaluate, then call hartmann6 using those parameters, and finally "tell" Ax the result using client.complete_trial.

This loop will run multiple trials to optimize the function.

In [ ]:
# 4. Conduct the experiment with 20 trials: get each trial from Ax, evaluate the
# objective function, and log data back to Ax.
number_of_experiments = 20
max_trials = 3

for _ in range(number_of_experiments):
    # We will request three trials at a time in this example
    trials = client.get_next_trials(max_trials=max_trials)

    for trial_index, parameters in trials.items():
        x1 = parameters["x1"]
        x2 = parameters["x2"]
        x3 = parameters["x3"]
        x4 = parameters["x4"]
        x5 = parameters["x5"]
        x6 = parameters["x6"]

        result = hartmann6(x1, x2, x3, x4, x5, x6)

        # Set raw_data as a dictionary with metric names as keys and results as values
        raw_data = {metric_name: result}

        # Complete the trial with the result
        client.complete_trial(trial_index=trial_index, raw_data=raw_data)

After running trials, you can analyze the results. Most commonly this means extracting the parameterization from the best performing trial you conducted. Hartmann6 has a known global minimum of $f(x*) = -3.322$ at $x^* = (0.201, 0.150, 0.477, 0.273, 0.312, 0.657)$. Ax is able to identify a point very near to this true optimum using just 30 evaluations. This is possible due to the sample-efficiency of Bayesian optimization, the optimization method we use under the hood in Ax.

In [ ]:
# 5. Obtain the best-performing configuration; the true minimum for the booth
best_parameters, prediction, index, name = client.get_best_parameterization()
print("Best Parameters:", best_parameters)
print("Prediction (mean, variance):", prediction)

In [ ]:
# display=True instructs Ax to sort then render the resulting analyses
cards = client.compute_analyses(display=True)

#Hyperparameter tuning using BO (AutoML)
Automated machine learning (AutoML) encompasses a large class of problems related to automating time-consuming and labor-intensive aspects of developing ML models. Adaptive experimentation is a natural fit for solving many AutoML tasks, which are often iterative in nature and can involve many expensive trial evaluations.

In this tutorial we will use Ax for hyperparameter optimization (HPO), a common AutoML task in which a model's hyperparameters are adjusted to improve model performance. Hyperparameters refer to the parameters which are set prior to model training or fitting, rather than parameters being learned from data. Traditionally, ML engineers use a combination of domain knowledge, intuition, and manual experimentation comparing many models with different hyperparameter configurations to determine good hyperparameters. As the number of hyperparameters grows and as models become more expensive to train and evaluate sample efficient aproaches to experimentation like Bayesian optimization become increasingly valuable.

In this tutorial we will train an SGDClassifier from the popular scikit-learn library to recognize handwritten digits and tune the model's hyperparameters to improve its performance.

## Learning Objectives
- Understand how Ax can be used for HPO tasks
- Use complex optimization configurations like multiple objectives and outcome - constraints to achieve nuanced real-world goals
- Use early stopping to save experimentation resources
- Analyze the results of the optimization

In [ ]:
import time

import matplotlib.pyplot as plt

import numpy as np

import sklearn.datasets
import sklearn.linear_model
import sklearn.model_selection

from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig

from pyre_extensions import assert_is_instance

Before we begin HPO, let's understand the task and the performance of SGDClassifier with its default hyperparameters.

In [ ]:
# Baseline performance of SGDClassifier
# Load the digits dataset and display the first 4 images to demonstrate
digits = sklearn.datasets.load_digits()
classes = list(set(digits.target))

_, axes = plt.subplots(nrows=1, ncols=4, figsize=(10, 3))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.set_axis_off()
    ax.imshow(image, cmap=plt.cm.gray_r, interpolation="nearest")
    ax.set_title("Training: %i" % label)

In [ ]:
# Split the data into a training set and a validation set
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    digits.data, digits.target, test_size=0.20, random_state=0
)

# Instantiate a SGDClassifier with default hyperparameters
clf = sklearn.linear_model.SGDClassifier()

# Train the classifier on the training set using 10 batches.
# Also time the training.
n_batches = 10
batch_size = X_train.shape[0] // n_batches
start_time = time.monotonic()
for epoch in range(20):
    for i in range(n_batches):
        start = i * batch_size
        end = start + batch_size

        X_batch = X_train[start:end]
        y_batch = y_train[start:end]

        clf.partial_fit(X_batch, y_batch, classes=classes)

training_time = time.monotonic() - start_time

# Evaluate the classifier on the validation set
score = clf.score(X_test, y_test)
print('model prformance with default hyperparameters')
print('score:',round(score*100,1),'%', 'traning_time:',round(training_time,1),'s')

The model performs well, but let's see if we can improve performance by tuning the hyperparameters.

In [ ]:
# Initialize the Client
client = Client()

The Client expects a series of Configs which define how the experiment will be run. We'll set this up the same way as we did in our previous tutorial.

Our current task is to tune the hyperparameters of an scikit-learn's SGDClassifier. These parameters control aspects of the model's training process and configuring them can have dramatic effects on the model's ability to correctly classify inputs. A full list of this model's hyperparameters and appropriate values are available in the library's documentation. In this tutorial we will tune the following hyperparameters:

- loss: The loss function to be used
- penalty: The penalty (aka regularization term) to be used
- learning_rate: The learning rate schedule
- alpha: Constant that multiplies the regularization term. The higher the value, the stronger the regularization
- eta0: The learning rate for training. In this example we will use a constant learning rate schedule
- batch_size: A training parameter which controls how many examples are shown during a single call to partial_fit

You will notice some hyperparameters are continuous ranges, some are discrete ranges, and some are categorical choices; Ax is able to handle all of these types of parameters via its RangeParameterConfig and ChoiceParameterConfig classes.

In [ ]:
# Configure and experiment with the desired parameters
client.configure_experiment(
    parameters=[
        ChoiceParameterConfig(
            name="loss",
            parameter_type="str",
            values=[
                "hinge",
                "log_loss",
                "squared_hinge",
                "modified_huber",
                "perceptron",
            ],
            is_ordered=False,
        ),
        ChoiceParameterConfig(
            name="penalty",
            parameter_type="str",
            values=["l1", "l2", "elasticnet"],
            is_ordered=False,
        ),
        ChoiceParameterConfig(
            name="learning_rate",
            parameter_type="str",
            values=["constant", "optimal", "invscaling", "adaptive"],
            is_ordered=False,
        ),
        RangeParameterConfig(
            name="alpha",
            bounds=(1e-8, 100),
            parameter_type="float",
            scaling="log",  # Sample this parameter in log transformed space
        ),
        RangeParameterConfig(
            name="eta0",
            bounds=(1e-8, 1),
            parameter_type="float",
            scaling="log",
        ),
        RangeParameterConfig(
            name="batch_size",
            bounds=(5, 500),
            parameter_type="int",
        ),
    ]
)

Now, we must set up the optimization objective in Client, where objective is a string that specifies which metric we would like to optimize and the direction (higher or lower) that is considered optimal.

In our example we want to consider both performance and computational cost implications of hyperparameter modifications. scikit-learn models use a function called score to report the mean accuracy of the model, and in our optimization we should seek to maximize this value. Since model training can be a very expensive process, especially for large models, this can represent a significant cost.

Let's configure Ax to maximize score while minimizing training time. We call this a multi-objective optimization, and rather than returning a single best parameterization we return a Pareto frontier of points which represent optimal tradeoffs between all metrics present. Multi-objective optimization is useful for competing metrics where a gain in one metric may represent a regression in the other.

In these settings we can also specify outcome constraints, which indicate that if a metric result falls outside of the specified threshold we are not interested in any result, regardless of the wins observed in any other metric. For a concrete example, imagine Ax finding a parameterization that trains in no time at all but has an score no better than if the model were guessing at random.

For this toy example let's configure Ax to maximize score and minimize training time, but avoid any hyperparameter configurations that result in a mean accuracy score of less than 85% or a training time greater than 2 seconds.

In [ ]:
# Configure Optimization
client.configure_optimization(
    objective="score, -training_time",
    outcome_constraints=["score >= 0.85", "training_time <= 2"],
)

Before we begin our Bayesian optimization loop, we can attach the data we collected from triaing SGDClassifier with default hyperparameters. This will give our experiment a head start by providing a datapoint to our surrogate model. Because these are the default settings provided by scikit-learn, it's likely they will be pretty good and will provide the optimization with a promising start. It is always advantageous to attach any existing data to an experiment to improve performance.

In [ ]:
# Run Trials with early stopping
trial_index = client.attach_baseline(
    parameters={
        "loss": clf.loss,
        "penalty": clf.penalty,
        "alpha": clf.alpha,
        "learning_rate": clf.learning_rate,
        "eta0": clf.eta0
        + 1e-8,  # Default eta is 0.0, so add a small value to avoid division by zero
        "batch_size": batch_size,
    }
)

client.complete_trial(
    trial_index=trial_index,
    raw_data={"score": score, "training_time": training_time},
)

After attaching the initial trial, we will begin the experimentation loop by writing a for loop to execute our full experimentation budget of 30 trials. In each iteration we will ask Ax for the next trials (in this case just one), then instantiate an SGDClassifier with the suggested hyperparameters. Next we will define two inner loops to perform minibatch training, in which we divide the train set into a number of smaller batches and train one epoch of stochastic gradient descent at a time. After each epoch we will report the score and the time.

Because training machine learning models is expensive, we will utilize Ax's early stopping functionality to kill trials unlikely to produce optimal results before they have been completed. After data has been attached we will ask the Client whether or not we should stop the trial, and if it advises us to do so we will report it early stopped and exit out of the training loop. By early stopping, we proactively save compute without regressing optimization performance.

In [ ]:
number_of_experiments = 20
max_trials = 1

for _ in range(number_of_experiments): # Run 20 rounds of 1 trial each
    trials = client.get_next_trials(max_trials=max_trials)
    for trial_index, parameters in trials.items():
        clf = sklearn.linear_model.SGDClassifier(
            loss=parameters["loss"],
            penalty=parameters["penalty"],
            alpha=parameters["alpha"],
            learning_rate=parameters["learning_rate"],
            eta0=parameters["eta0"],
        )

        batch_size = assert_is_instance(parameters["batch_size"], int)
        n_batches = X_train.shape[0] // batch_size

        # Create a flag to track whether the trial was stopped early
        is_early_stopped = False

        start_time = time.monotonic()
        for epoch in range(20):
            for i in range(n_batches):
                # Calculate the minibatch and fit the model
                start = i * batch_size
                end = start + batch_size

                X_batch = X_train[start:end]
                y_batch = y_train[start:end]

                clf.partial_fit(X_batch, y_batch, classes=classes)

            # Evaluate the model and report the score back to Ax
            score = clf.score(X_test, y_test)

            client.attach_data(
                trial_index=trial_index,
                raw_data={"score": score, "training_time": time.monotonic() - start_time},
                progression=epoch + 1,
            )

            # If the trial is underperforming, stop it
            if client.should_stop_trial_early(trial_index=trial_index):
                is_early_stopped = True
                client.mark_trial_early_stopped(trial_index=trial_index)
                break

        if is_early_stopped:
            break


        if not is_early_stopped:
            client.complete_trial(trial_index=trial_index)


After running trials, you can analyze the results. Most commonly this means extracting the parameterization from the best performing trial you conducted.

Since we are optimizing multiple objectives, rather than a single best point we want to get the Pareto frontier -- the set of points that presents optimal tradeoffs between maximizing score and minimizing training time.

In [ ]:
#Analyze Results
frontier = client.get_pareto_frontier()

# Frontier is a list of tuples, where each tuple contains the parameters, the metric readings, the trial index, and the arm name for a point on the Pareto frontier
for parameters, metrics, trial_index, arm_name in frontier:
    print(f"Trial {trial_index} with {parameters=} and {metrics=}\n")

In [ ]:
# display=True instructs Ax to sort then render the resulting analyses
cards = client.compute_analyses(display=True)

# Material experiment using BO (Automated exeperiment)
Some optimization experiments, like the one described in this tutorial, can be conducted in a completely automated manner. Other experiments may require a human in the loop, for instance a scientist manually conducting and evaluating each trial in a lab. In this tutorial we demonstrate this ask-tell optimization in a human-in-the-loop setting by imagining the task of maximizing the strength of a 3D printed part using compression testing (i.e., crushing the part) where different print settings will have to be manually tried and evaluated.

## Background
In 3D printing, several parameters can significantly affect the mechanical properties of the printed object:

- Infill Density: The percentage of material used inside the object. Higher infill density generally increases strength but also weight and material usage.

- Layer Height: The thickness of each layer of material. Smaller layer heights can improve surface finish and detail but increase print time.

- Infill Type: The pattern used to fill the interior of the object. Different patterns (e.g., honeycomb, gyroid, lines, rectilinear) offer various balances of strength, speed, and material efficiency.

- Strength Measurement: In this tutorial, we assume the strength of the 3D printed part is measured using compression testing, which evaluates how the object performs under compressive stress.

In [ ]:
from ax.api.client import Client
from ax.api.configs import  RangeParameterConfig, ChoiceParameterConfig

In [ ]:
# Initialize Client
client = Client()

Define the parameters for the 3D printing optimization problem. The infill density and layer height can take on any value within their respective bounds so we will configure both using RangeParameterConfigs. On the other hand, infill type be either have one of four distinct values: "honeycomb", "gyroid", "lines", or "rectilinear". We will use a ChoiceParameterConfig to represent it in the optimization.

In [ ]:
# Configure Experiment
infill_density = RangeParameterConfig(name="infill_density", parameter_type="float", bounds=(0, 100))
layer_height = RangeParameterConfig(name="layer_height", parameter_type="float", bounds=(0.1, 0.4))
infill_type = ChoiceParameterConfig(name="infill_type", parameter_type="str", values=["honeycomb", "gyroid", "lines", "rectilinear"])

client.configure_experiment(
    parameters=[infill_density, layer_height, infill_type],
    # The following arguments are only necessary when saving to the DB
    name="3d_print_strength_experiment",
    description="Maximize strength of 3D printed parts",
    owner="developer",
)

We want to maximize the compressive strength of our part, so we will set the objective to compressive_strength. However, we know that modifying the infill density, layer height, and infill type will affect the weight of the part as well. We'll include a requirement that the part must not weigh more than 10 grams by setting an outcome constraint when we call configure_experiment.

The following code will tell the Client that we intend to maximize compressive strength while keeping the weight less than 10 grams.

In [ ]:
# Configure Optimization
client.configure_optimization(objective="compressive_strength", outcome_constraints=["weight <= 10"])

Sometimes in our optimization experiments we may already have some previously collected data from manual "trials" conducted before the Ax experiment began. This can be incredibly useful! If we attach this data as custom trials, Ax will be able to use the data points in its optimization algorithm and improve performance.

In [ ]:
# Pairs of previously evaluated parameterizations and associated metric readings
preexisting_trials = [
    (
        {"infill_density": 10.43, "layer_height": 0.3, "infill_type": "gyroid"},
        {"compressive_strength": 1.74, "weight": 0.52},
    ),
    (
        {"infill_density": 55.54, "layer_height": 0.12, "infill_type": "lines"},
        {"compressive_strength": 4.63, "weight": 2.31},
    ),
    (
        {"infill_density": 99.43, "layer_height": 0.35, "infill_type": "rectilinear"},
        {"compressive_strength": 5.68, "weight": 2.84},
    ),
    (
        {"infill_density": 41.44, "layer_height": 0.21, "infill_type": "rectilinear"},
        {"compressive_strength": 3.95, "weight": 1.97},
    ),
    (
        {"infill_density": 27.23, "layer_height": 0.37, "infill_type": "honeycomb"},
        {"compressive_strength": 7.36, "weight": 3.31},
    ),
    (
        {"infill_density": 33.57, "layer_height": 0.24, "infill_type": "honeycomb"},
        {"compressive_strength": 13.99, "weight": 6.29},
    ),
]

for parameters, data in preexisting_trials:
    # Attach the parameterization to the Client as a trial and immediately complete it with the preexisting data
    trial_index = client.attach_trial(parameters=parameters)
    client.complete_trial(trial_index=trial_index, raw_data=data)

Now, let's have Ax suggest which trials to evaluate so that we can find the optimal configuration more efficiently. We'll do this by calling get_next_trials. We'll make use of Ax's support for parallelism, i.e. suggesting more than one trial at a time -- this can allow us to conduct our experiment much faster! If our lab had three identical 3D printers, we could ask Ax for a batch of three trials and evaluate three different infill density, layer height, and infill types at once.

Note that there will always be a tradeoff between "parallelism" and optimization performance since the quality of a suggested trial is often proportional to the amount of data Ax has access to.

In [ ]:
# Ask for trials
trials = client.get_next_trials(max_trials=3)
trials

In a real-world scenerio we would print parts using the three suggested parameterizations and measure the compressive strength and weight manually, though in this tutorial we will simulate by calling a function. Once the data is collected we will tell Ax the result by calling complete_trial.

In [ ]:
# Tell Ax the results
def evaluate(
    infill_density: float, layer_height: float, infill_type: str
) -> dict[str, float]:
    strength_map = {"lines": 1, "rectilinear": 2, "gyroid": 5, "honeycomb": 10}
    weight_map = {"lines": 1, "rectilinear": 2, "gyroid": 3, "honeycomb": 9}

    return {
        "compressive_strength": (
            infill_density / layer_height * strength_map[infill_type]
        )
        / 100,
        "weight": (infill_density / layer_height * weight_map[infill_type]) / 200,
    }


for trial_index, parameters in trials.items():
    client.complete_trial(trial_index=trial_index, raw_data=evaluate(**parameters))

We'll repeat this process a number of times. Typically experimentation will continue until a satisfactory combination has been found, experimentation resources (in this example our 3D printing filliment) have been exhausted, or we feel we have spent enough time on optimization.

In [ ]:
# Ask Ax for the next trials
trials = client.get_next_trials(max_trials=3)
trials

In [ ]:
# Tell Ax the result of those trials
for trial_index, parameters in trials.items():
    client.complete_trial(trial_index=trial_index, raw_data=evaluate(**parameters))

In [ ]:
# Ask Ax for the next trials
trials = client.get_next_trials(max_trials=3)
trials

In [ ]:
# Tell Ax the result of those trials
for trial_index, parameters in trials.items():
    client.complete_trial(trial_index=trial_index, raw_data=evaluate(**parameters))

In [ ]:
# Ask Ax for the next trials
trials = client.get_next_trials(max_trials=3)
trials

At any time during the experiment you may analyze the results of the experiment. Most commonly this means extracting the parameterization from the best performing trial you conducted. The best trial will have the optimal objective value without violating any outcome constraints.

In [ ]:
best_parameters, prediction, index, name = client.get_best_parameterization()
print("Best Parameters:", best_parameters)
print("Prediction (mean, variance):", prediction)

In [ ]:
# display=True instructs Ax to sort then render the resulting analyses
cards = client.compute_analyses(display=True)